In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Model
# Configuración del Architect
BUCKET_NAME = "sentinel-vision-data-v1" 

# Necesitamos recrear la lista de clases para que el modelo sepa cuántas salidas tener
class_names = ['Forest', 'Highway', 'Industrial', 'Land', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'Sea', 'Vegetation'] 
class_names.sort()

2026-04-29 21:29:11.465130: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-29 21:29:16.980154: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/lib/x86_64-linux-gnu/:/opt/conda/lib
2026-04-29 21:29:16.980669: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local

In [3]:
print("GPUs disponibles: ", len(tf.config.list_physical_devices('GPU')))

GPUs disponibles:  0


2026-04-29 21:29:23.314292: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/lib/x86_64-linux-gnu/:/opt/conda/lib
2026-04-29 21:29:23.316305: W tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:265] failed call to cuInit: UNKNOWN ERROR (303)
2026-04-29 21:29:23.316425: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (sentinel-vision-dev): /proc/driver/nvidia/version does not exist


In [3]:
# Definimos la entrada de forma explícita
# Aquí creamos el "molde" del que hablábamos
inputs = layers.Input(shape=(224, 224, 3), name="input_satelital")

In [4]:
# 1. Cargar el modelo base pre-entrenado
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False, # Quitamos la cabeza original de 1000 clases
    weights='imagenet' # Usamos el conocimiento de Google
)

# 2. Congelar el modelo base (No queremos que cambie lo que ya sabe)
base_model.trainable = False

# 3. Definir la estructura funcional
inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False) # Pasamos la entrada por la base
x = layers.GlobalAveragePooling2D()(x) # Aplanamos el cubo
outputs = layers.Dense(len(class_names), activation='softmax')(x) # Capa final de decisión

# 4. Crear el modelo
model = Model(inputs, outputs, name="Sentinel_Vision_Alpha")

model.summary()

2026-04-27 02:04:09.820780: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/lib/x86_64-linux-gnu/:/opt/conda/lib
2026-04-27 02:04:09.820824: W tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:265] failed call to cuInit: UNKNOWN ERROR (303)
2026-04-27 02:04:09.820853: I tensorflow/compiler/xla/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (sentinel-vision-dev): /proc/driver/nvidia/version does not exist
2026-04-27 02:04:09.821102: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in 

9406464/9406464 [==============================] - 0s 0us/step
Model: "Sentinel_Vision_Alpha"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 mobilenetv2_1.00_224 (Funct  (None, 7, 7, 1280)       2257984   
 ional)                                                          
                                                                 
 global_average_pooling2d (G  (None, 1280)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 10)                12810     
                                                                 
Total params: 2,270,794
Trainable params: 12,810
Non-trainable params: 2,257,984
_________________________________

In [5]:
# Definimos el optimizador, la función de pérdida y las métricas
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy', # Usamos esta porque nuestras etiquetas son números enteros (0, 1, 2...)
    metrics=['accuracy']
)

print("Cerebro configurado y listo para aprender.")

Cerebro configurado y listo para aprender.


In [9]:
import os
from tensorflow.keras.layers import StringLookup
# 1. Definir la ruta de tus datos en el Bucket
GCS_PATH = f"gs://{BUCKET_NAME}/data/*/*.jpg"

label_encoder = StringLookup(vocabulary=class_names, mask_token=None)
# 2. El comando que crea el "cubo de rutas" (list_ds)
list_ds = tf.data.Dataset.list_files(GCS_PATH, shuffle=True)

# 3. La función de transformación (la que decodifica y normaliza)
def process_path_with_label(file_path):
    label_text = tf.strings.split(file_path, os.path.sep)[-2]
    label_idx = label_encoder(label_text) # Usa el StringLookup que definimos antes
    
    img = tf.io.read_file(file_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label_idx

# 4. Crear el flujo final (Pipeline)
BATCH_SIZE = 32
train_ds = list_ds.map(process_path_with_label, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Misión: Entrenamiento del Cerebro Satelital
print("Iniciando entrenamiento en Vertex AI...")

# Guardamos el resultado en 'history' para graficar después
history = model.fit(
    train_ds,           # Tu pipeline de datos (el flujo de cubos 4D)
    epochs=5,           # 5 vueltas completas al dataset
    verbose=1           # Para que veamos la barrita de progreso
)

print("\n¡Entrenamiento completado, Ingeniera!")

Iniciando entrenamiento en Vertex AI...
Epoch 1/5
141/844 [====>.........................] - ETA: 9:32 - loss: 0.7362 - accuracy: 0.7522